# Critic ablation: return-normalization + Huber + value-clip

Сравнение PPO-критика **без** и **с** улучшениями (п.1 нормализация таргетов +
п.2 Huber + value-clip). Обе модели обучаются на **одной** задаче
(connectivity-only + `cap`, W=10), одинаковое число эпох и тот же сплит — отличаются
**только флаги критика** (`critic_normalize_returns` / `critic_huber` /
`critic_value_clip`, по умолчанию OFF).

Главная метрика — **`explained_variance`** критика: scale-free (1 − Var(резидуала)/Var(таргета)),
сравнима между режимами. **Внимание:** `critic_mse_mean` у нового режима считается в
**нормированном** пространстве (Huber-loss), поэтому абсолютные значения loss между
режимами НЕ сравнимы — ориентируйтесь на explained_variance.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from IPython.display import display

from eval_lib.context import ROOT_DIR, CFG_DIR, DATASETS_DIR, MODEL_OUTPUTS_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    load_raw_graphs_and_lc_routes, train_lc_improvement_cfg)
from connectpt.routes_generator.citygraph_dataset import STOP_KEY
from eval_lib.results_io import save_table

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Конфигурация эксперимента

In [ ]:
# Небольшой эксперимент: подмножество графов, 20 эпох на вариант.
DATASET_DIRNAME = "mixed_conn_adj_n50_r12_len8_15"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL      = NEW_DATASET_DIR / "raw_graphs_subset.pkl"

EXP_N_GRAPHS = 150          # подмножество для скорости
N_ITERATIONS = 20           # эпох на каждый вариант критика
BATCH_SIZE   = 16
VAL_PERIOD   = 2
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0            # одинаковый сплит/сиды -> отличается ТОЛЬКО критик

TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15

# Задача: connectivity-only + cap (тяжёлые хвосты returns из-за штрафа -> хороший
# стресс-тест для критика). Без conditioning, чтобы сравнение было чистым.
DISABLED      = ["demand", "route"]
ADJ_OBJECTIVE = "cap"
ADJ_WEIGHT    = 10.0
ADJ_TARGET    = 0.15

# Варианты критика: tag -> доп. cfg-оверрайды (по умолчанию всё OFF = старый критик).
CRITIC_VARIANTS = {
    "baseline (MSE, no-norm)": [],
    "norm+huber+clip": [
        "++critic_normalize_returns=true",
        "++critic_huber=true",
        "++critic_huber_delta=1.0",
        "++critic_value_clip=0.2",
    ],
}
print(f"{EXP_N_GRAPHS} graphs x {N_ITERATIONS} epochs, variants: {list(CRITIC_VARIANTS)}")

## Датасет (подмножество) + split

In [ ]:
graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
graphs = graphs[:EXP_N_GRAPHS]
seed_routes = seed_routes[:EXP_N_GRAPHS]
N_GRAPHS = len(graphs)
_perm = torch.randperm(N_GRAPHS,
                       generator=torch.Generator().manual_seed(SPLIT_SEED))
_n_train = int(TRAIN_FRACTION * N_GRAPHS)
TRAIN_INDICES = _perm[:_n_train].clone()
VAL_INDICES = _perm[_n_train:].clone()
print(f"graphs={N_GRAPHS} | train={len(TRAIN_INDICES)} val={len(VAL_INDICES)} "
      f"| node-feat x={tuple(graphs[0][STOP_KEY].x.shape)}")

## Обучение двух вариантов критика

In [ ]:
def train_critic_variant(tag, critic_overrides):
    """Обучить агента с заданными флагами критика; вернуть history DataFrame.
    Одинаковая задача/сплит/сиды -> разница только в критике."""
    ov = [
        "model=bestsofar_feb2023_trim",
        "model.route_generator.kwargs.serial_halting=True",
        "++run_name=critic_exp",
        "++experiment.logdir=null",
        f"++adjustment_degree_weight={float(ADJ_WEIGHT)}",
        f"++adjustment_degree_target={float(ADJ_TARGET)}",
        f"++adjustment_degree_objective={ADJ_OBJECTIVE}",
    ] + list(critic_overrides)
    with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
        cfg = compose(config_name="ppo_50nodes.yaml", overrides=ov)
    _, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg, run_name_prefix="critic_")
    cost_obj.ignore_stops_oob = True
    cost_obj.set_enabled_components(disabled_components=DISABLED or None)
    safe = tag.replace(" ", "_").replace("(", "").replace(")", "").replace(
        ",", "").replace("+", "_").replace("-", "_")
    res = train_lc_improvement_cfg(
        model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
        device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR,
        run_name=f"critic_exp_{safe}", train_fraction=TRAIN_FRACTION,
        batch_size=BATCH_SIZE, min_route_len=MIN_ROUTE_LEN,
        max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
        target_n_routes=TARGET_N_ROUTES,
        train_indices=TRAIN_INDICES, val_indices=VAL_INDICES,
        best_model_path=MODEL_OUTPUTS_DIR / f"critic_exp_{safe}.pt",
        n_iterations=N_ITERATIONS, val_period=VAL_PERIOD,
    )
    return pd.DataFrame(res["history"])


histories = {}
for _tag, _ov in CRITIC_VARIANTS.items():
    print(f"\n=== training: {_tag} ===")
    histories[_tag] = train_critic_variant(_tag, _ov)
print("done")

## Кривые обучения критика

`explained_variance` (слева) — единственная **сравнимая** между режимами метрика.
Loss (центр) у нового режима в нормированном Huber-пространстве, поэтому его кривая
ниже не из-за «лучшести», а из-за другого масштаба — сравнивать абсолютные значения
нельзя. `val_delta` (справа) — влияние на качество политики.

In [ ]:
def _num(h, col):
    return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6), constrained_layout=True)
for tag, h in histories.items():
    ep = h["epoch"]
    ev = _num(h, "train_critic_explained_variance")
    if ev is not None:
        axes[0].plot(ep, ev, marker="o", ms=3, label=tag)
    ms = _num(h, "train_critic_mse_mean")
    if ms is not None:
        axes[1].plot(ep, ms, marker="o", ms=3, label=tag)
    vd = _num(h, "val_delta")
    if vd is not None and vd.notna().any():
        axes[2].plot(ep, vd, marker="o", ms=3, label=tag)

axes[0].axhline(0, color="k", lw=1)
axes[0].set_title("critic explained_variance (выше=лучше; СРАВНИМО)")
axes[1].set_title("critic loss (в своём пространстве; НЕ сравнимо)")
axes[2].axhline(0, color="k", lw=1)
axes[2].set_title("val cost delta (+ = улучшение)")
for a in axes:
    a.set_xlabel("epoch"); a.grid(alpha=0.25); a.legend(fontsize=8)
fig.suptitle("Critic ablation: with vs without normalization+Huber+value-clip",
             fontsize=13, fontweight="bold")
plt.show()
plt.close(fig)

## Сводка

In [ ]:
rows = []
for tag, h in histories.items():
    ev = _num(h, "train_critic_explained_variance")
    vd = _num(h, "val_delta")
    rows.append({
        "variant": tag,
        "final_explained_var": float(ev.dropna().iloc[-1]) if ev is not None and ev.notna().any() else float("nan"),
        "mean_last5_explained_var": float(ev.dropna().tail(5).mean()) if ev is not None and ev.notna().any() else float("nan"),
        "final_val_delta": float(vd.dropna().iloc[-1]) if vd is not None and vd.notna().any() else float("nan"),
    })
summary = pd.DataFrame(rows).round(3)
display(summary)
save_table(summary, "critic_ablation_summary")
print("Saved -> artifacts/results/critic_ablation_summary.csv")